# Crop Health Disease Detection Pegasus Workflow

This workflow processes crop images through a CNN-based disease detection pipeline using Pegasus WMS on ACCESS resources. It fetches crop images, preprocesses them for machine learning, trains a disease classification model, runs inference, and generates a comprehensive health report with visualizations and treatment recommendations.

**Container:** `kthare10/crophealth:latest` from DockerHub, containing PyTorch, NumPy, Pandas, Pillow, Matplotlib, and scikit-learn.

**Workflow Jobs:**
1. `fetch_crop_images` - Fetch and catalog crop images from local directory, Kaggle, or sample data
2. `preprocess_images` - Resize, normalize, augment images and split into train/validation sets
3. `train_classifier` - Train a CNN disease classifier using PyTorch (with sklearn fallback)
4. `classify_disease` - Run inference on images and generate disease predictions with confidence scores
5. `generate_report` - Create HTML report, summary JSON, and visualization charts

**Data Sources:**
- `sample` - Generated sample data for quick testing (no external dependencies)
- `kaggle` - PlantVillage dataset from Kaggle (requires API key)
- `local` - Pre-downloaded local image directory

## Install Dependencies

In [ ]:
!pip3 install pandas

## 1. Configure Workflow Parameters

Update the parameters below to customize the workflow. For quick testing, use `sample` as the data source. For real experiments, use `kaggle` to download the PlantVillage dataset or `local` with a pre-downloaded image directory.

In [ ]:
# Data source: "sample", "kaggle", or "local"
DATA_SOURCE = "kaggle"

# Kaggle dataset (only used when DATA_SOURCE = "kaggle")
KAGGLE_DATASET = "emmarex/plantdisease"

# Local image directory (only used when DATA_SOURCE = "local")
IMAGE_DIR = "./field_images"

# Image preprocessing parameters
IMAGE_SIZE = 128        # Target image size (square)
TRAIN_SPLIT = 0.8       # Fraction of data for training

# Model training parameters
EPOCHS = 10             # Number of training epochs
BATCH_SIZE = 16         # Training batch size

## Kaggle API Key

If using `kaggle` as the data source, you need a Kaggle API key:

1. Create an account at [https://www.kaggle.com/](https://www.kaggle.com/)
2. Go to Settings > API > Create New Token
3. Accept the dataset terms at [https://www.kaggle.com/datasets/emmarex/plantdisease](https://www.kaggle.com/datasets/emmarex/plantdisease)
4. Set your credentials below

In [ ]:
import os

os.environ["KAGGLE_USERNAME"] = "YOUR_KAGGLE_USERNAME"
os.environ["KAGGLE_KEY"] = "YOUR_KAGGLE_API_KEY"

## 2. Create the Crop Health Workflow

The following cell defines the `CropHealthWorkflow` class, which builds the Pegasus workflow DAG with site, transformation, and replica catalogs. It then generates and writes the workflow file.

In [ ]:
import os
import sys
import logging
import argparse
from pathlib import Path

from Pegasus.api import *

logging.basicConfig(level=logging.DEBUG)


class CropHealthWorkflow:
    """Generate Pegasus workflow for crop disease detection."""

    wf = None
    sc = None
    tc = None
    rc = None
    props = None

    dagfile = None
    wf_dir = None
    shared_scratch_dir = None
    local_storage_dir = None
    wf_name = "crophealth"

    def __init__(self, dagfile="workflow.yml"):
        """Initialize workflow."""
        self.dagfile = dagfile
        self.wf_dir = str(Path(".").resolve())
        self.shared_scratch_dir = os.path.join(self.wf_dir, "scratch")
        self.local_storage_dir = os.path.join(self.wf_dir, "output")

    def write(self):
        """Write all catalogs and workflow to files."""
        if self.sc is not None:
            self.sc.write()
        self.props.write()
        self.rc.write()
        self.tc.write()
        try:
            self.wf.write(file=self.dagfile)
        except PegasusClientError as e:
            print(e)

    def plan_submit(self):
        """Plan and submit the workflow."""
        try:
            self.wf.plan(submit=True)
        except PegasusClientError as e:
            print(e)

    def status(self):
        """Get workflow status."""
        try:
            self.wf.status(long=True)
        except PegasusClientError as e:
            print(e)

    def wait(self):
        """Wait for workflow completion."""
        try:
            self.wf.wait()
        except PegasusClientError as e:
            print(e)

    def statistics(self):
        """Get workflow statistics."""
        try:
            self.wf.statistics()
        except PegasusClientError as e:
            print(e)

    def create_pegasus_properties(self):
        """Create Pegasus properties configuration."""
        self.props = Properties()
        self.props["pegasus.transfer.threads"] = "16"

    def create_sites_catalog(self, exec_site_name="condorpool"):
        """Create site catalog."""
        self.sc = SiteCatalog()

        local = Site("local").add_directories(
            Directory(
                Directory.SHARED_SCRATCH, self.shared_scratch_dir
            ).add_file_servers(
                FileServer("file://" + self.shared_scratch_dir, Operation.ALL)
            ),
            Directory(
                Directory.LOCAL_STORAGE, self.local_storage_dir
            ).add_file_servers(
                FileServer("file://" + self.local_storage_dir, Operation.ALL)
            ),
        )

        exec_site = (
            Site(exec_site_name)
            .add_condor_profile(universe="vanilla")
            .add_pegasus_profile(style="condor")
        )

        self.sc.add_sites(local, exec_site)

    def create_replica_catalog(self):
        """Create replica catalog for input files."""
        self.rc = ReplicaCatalog()

    def create_transformation_catalog(self, exec_site_name="condorpool", container_image="kthare10/crophealth:latest"):
        """Create transformation catalog with executables and containers."""
        self.tc = TransformationCatalog()

        crophealth_container = Container(
            "crophealth_container",
            container_type=Container.SINGULARITY,
            image=f"docker://{container_image}",
            image_site="docker_hub",
        )

        fetch_crop_images = Transformation(
            "fetch_crop_images",
            site=exec_site_name,
            pfn=os.path.join(self.wf_dir, "fetch_crop_images.py"),
            is_stageable=True,
            container=crophealth_container,
        ).add_pegasus_profile(memory="4 GB")

        preprocess_images = Transformation(
            "preprocess_images",
            site=exec_site_name,
            pfn=os.path.join(self.wf_dir, "bin/preprocess_images.py"),
            is_stageable=True,
            container=crophealth_container,
        ).add_pegasus_profile(memory="32 GB")

        train_classifier = Transformation(
            "train_classifier",
            site=exec_site_name,
            pfn=os.path.join(self.wf_dir, "bin/train_classifier.py"),
            is_stageable=True,
            container=crophealth_container,
        ).add_pegasus_profile(memory="32 GB")

        classify_disease = Transformation(
            "classify_disease",
            site=exec_site_name,
            pfn=os.path.join(self.wf_dir, "bin/classify_disease.py"),
            is_stageable=True,
            container=crophealth_container,
        ).add_pegasus_profile(memory="32 GB")

        generate_report = Transformation(
            "generate_report",
            site=exec_site_name,
            pfn=os.path.join(self.wf_dir, "bin/generate_report.py"),
            is_stageable=True,
            container=crophealth_container,
        ).add_pegasus_profile(memory="32 GB")

        self.tc.add_containers(crophealth_container)
        self.tc.add_transformations(
            fetch_crop_images,
            preprocess_images,
            train_classifier,
            classify_disease,
            generate_report,
        )

    def create_workflow(self, args):
        """Create the complete workflow."""
        self.wf = Workflow(self.wf_name)

        # Output file names
        catalog_file = File("crop_catalog.csv")
        images_archive = File("images.tar.gz")
        train_data = File("train_data.npz")
        val_data = File("val_data.npz")
        label_mapping = File("label_mapping.json")
        preprocessing_info = File("preprocessing_info.json")
        model_checkpoint = File("disease_classifier.pt")
        training_info = File("training_info.json")
        predictions_file = File("predictions.json")
        report_html = File("report.html")
        report_summary = File("report_summary.json")
        disease_distribution_png = File("disease_distribution.png")
        severity_distribution_png = File("severity_distribution.png")
        crop_health_summary_png = File("crop_health_summary.png")
        confidence_histogram_png = File("confidence_histogram.png")

        # Job 1: Fetch/catalog images
        fetch_job = Job("fetch_crop_images", _id="fetch_images", node_label="fetch_images")
        fetch_job.add_args(
            "--source", args.data_source,
            "--output", catalog_file,
            "--output-dir", "./images",
            "--archive-output", images_archive,
        )
        if args.data_source == "local" and args.image_dir:
            fetch_job.add_args("--input-dir", args.image_dir)
        elif args.data_source == "kaggle":
            fetch_job.add_args("--dataset", args.kaggle_dataset)
        fetch_job.add_env(KAGGLE_USERNAME=os.environ.get('KAGGLE_USERNAME', ''))
        fetch_job.add_env(KAGGLE_KEY=os.environ.get('KAGGLE_KEY', ''))
        fetch_job.add_outputs(catalog_file, stage_out=True, register_replica=False)
        fetch_job.add_outputs(images_archive, stage_out=False, register_replica=False)
        fetch_job.add_pegasus_profile(label="fetch")

        # Job 2: Preprocess images
        preprocess_job = Job("preprocess_images", _id="preprocess", node_label="preprocess")
        preprocess_job.add_args(
            "--input", catalog_file,
            "--output-dir", ".",
            "--image-size", str(args.image_size),
            "--split", str(args.train_split),
            "--images-archive", images_archive,
        )
        preprocess_job.add_inputs(catalog_file, images_archive)
        preprocess_job.add_outputs(train_data, stage_out=True, register_replica=False)
        preprocess_job.add_outputs(val_data, stage_out=True, register_replica=False)
        preprocess_job.add_outputs(label_mapping, stage_out=True, register_replica=False)
        preprocess_job.add_outputs(preprocessing_info, stage_out=True, register_replica=False)
        preprocess_job.add_pegasus_profile(label="preprocess")

        # Job 3: Train classifier
        train_job = Job("train_classifier", _id="train", node_label="train")
        train_job.add_args(
            "--input-dir", ".",
            "--output-dir", ".",
            "--epochs", str(args.epochs),
            "--batch-size", str(args.batch_size),
        )
        train_job.add_inputs(train_data, val_data, label_mapping)
        train_job.add_outputs(model_checkpoint, stage_out=True, register_replica=False)
        train_job.add_outputs(training_info, stage_out=True, register_replica=False)
        train_job.add_pegasus_profile(label="train")

        # Job 4: Classify diseases
        classify_job = Job("classify_disease", _id="classify", node_label="classify")
        classify_job.add_args(
            "--model-dir", ".",
            "--input", "./images",
            "--output", predictions_file,
            "--images-archive", images_archive,
        )
        classify_job.add_inputs(model_checkpoint, training_info, images_archive)
        classify_job.add_outputs(predictions_file, stage_out=True, register_replica=False)
        classify_job.add_pegasus_profile(label="classify")

        # Job 5: Generate report
        report_job = Job("generate_report", _id="report", node_label="report")
        report_job.add_args(
            "--predictions", predictions_file,
            "--output-dir", ".",
            "--format", "all",
        )
        report_job.add_inputs(predictions_file)
        report_job.add_outputs(report_html, stage_out=True, register_replica=False)
        report_job.add_outputs(report_summary, stage_out=True, register_replica=False)
        report_job.add_outputs(disease_distribution_png, stage_out=True, register_replica=False)
        report_job.add_outputs(severity_distribution_png, stage_out=True, register_replica=False)
        report_job.add_outputs(crop_health_summary_png, stage_out=True, register_replica=False)
        report_job.add_outputs(confidence_histogram_png, stage_out=True, register_replica=False)
        report_job.add_pegasus_profile(label="report")

        # Add jobs to workflow
        self.wf.add_jobs(fetch_job, preprocess_job, train_job, classify_job, report_job)

        # Define dependencies
        self.wf.add_dependency(fetch_job, children=[preprocess_job])
        self.wf.add_dependency(preprocess_job, children=[train_job])
        self.wf.add_dependency(train_job, children=[classify_job])
        self.wf.add_dependency(classify_job, children=[report_job])


# --- Build and generate the workflow ---
dagfile = 'workflow.yml'

args = argparse.Namespace(
    data_source=DATA_SOURCE,
    image_dir=IMAGE_DIR,
    kaggle_dataset=KAGGLE_DATASET,
    image_size=IMAGE_SIZE,
    train_split=TRAIN_SPLIT,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
)

workflow = CropHealthWorkflow(dagfile=dagfile)

print("Creating execution sites...")
workflow.create_sites_catalog("condorpool")

print("Creating workflow properties...")
workflow.create_pegasus_properties()

print("Creating transformation catalog...")
workflow.create_transformation_catalog("condorpool")

print("Creating replica catalog...")
workflow.create_replica_catalog()

print("Creating crop health workflow DAG...")
workflow.create_workflow(args)

workflow.write()
print("\nCrop Health Workflow has been generated!")

## View the Generated Workflow DAG

Before submitting, visualize the workflow DAG using `pegasus-graphviz`. The graph shows the linear pipeline: fetch images, preprocess, train classifier, classify diseases, and generate report.

In [ ]:
!pegasus-graphviz -f workflow.yml --output workflow.png

In [ ]:
from IPython.display import Image
Image(filename='workflow.png')

## 3. Plan and Submit the Workflow

The workflow will be planned and submitted for execution on the **condorpool** site (the selected ACCESS resource).

In [ ]:
workflow.plan_submit()

## Workflow Status Monitoring

After successful submission, monitor the workflow status. The output shows job counts and idle/running/completed states.

In [ ]:
workflow.status()

In [ ]:
workflow.wait()

## 4. Statistics

After the workflow completes, pull execution statistics from the Pegasus provenance database.

In [ ]:
workflow.statistics()

## 5. Examining the Results

The workflow produces the following outputs in the `output/` directory:

| File | Description |
|------|-------------|
| `crop_catalog.csv` | Image catalog with paths and categories |
| `train_data.npz` | Preprocessed training data (images + labels) |
| `val_data.npz` | Preprocessed validation data |
| `label_mapping.json` | Disease category to label index mapping |
| `preprocessing_info.json` | Preprocessing statistics and configuration |
| `disease_classifier.pt` | Trained PyTorch model checkpoint |
| `training_info.json` | Training metrics and history |
| `predictions.json` | Disease predictions with confidence scores and treatment recommendations |
| `report.html` | Interactive HTML report with all findings |
| `report_summary.json` | Summary statistics in JSON format |
| `disease_distribution.png` | Disease distribution chart |
| `severity_distribution.png` | Severity breakdown chart |
| `crop_health_summary.png` | Crop-wise health summary |
| `confidence_histogram.png` | Model confidence histogram |

In [ ]:
!ls -ltR output/

### Disease Detection Charts

The report generates four visualizations: disease distribution across the dataset, severity breakdown, crop-wise health summary, and a confidence histogram showing model prediction quality.

In [ ]:
import glob
from IPython.display import Image, display

chart_pngs = sorted(glob.glob("output/*.png"))
for png in chart_pngs:
    print(f"\n{png}")
    display(Image(filename=png))

### Report Summary

The JSON summary provides an overview of disease detection results including total predictions, healthy vs diseased counts, disease breakdown, and critical alerts.

In [ ]:
import json

summary_file = "output/report_summary.json"
try:
    with open(summary_file, 'r') as f:
        summary = json.load(f)

    stats = summary.get('summary', {})
    print(f"Total predictions:  {stats.get('total', 'N/A')}")
    print(f"Healthy:            {stats.get('healthy', 'N/A')}")
    print(f"Diseased:           {stats.get('diseased', 'N/A')}")
    print(f"\nDisease breakdown:")
    for disease, count in stats.get('disease_breakdown', {}).items():
        print(f"  {disease}: {count}")

    critical = summary.get('critical_alerts', [])
    if critical:
        print(f"\nCritical alerts: {len(critical)}")
        for alert in critical[:5]:
            print(f"  {alert.get('image', 'N/A')}: {alert.get('disease', 'N/A')} - {alert.get('action', 'N/A')}")
except FileNotFoundError:
    print(f"Summary file not found: {summary_file}")

### HTML Report

A comprehensive HTML report is generated at `output/report.html` with summary statistics, critical alerts, disease distribution charts, severity breakdown, treatment recommendations, and a detailed results table. Open it in a browser to view the full interactive report.